In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from ipywidgets import interact, Dropdown
import ipywidgets as widgets

In [10]:
data = pd.read_csv('HoC-GE2024-results-by-constituency.csv')
data.head()

,ONS ID,ONS region ID,Constituency name,County name,Region name,Country name,Constituency type,Declaration time,Member first name,Member surname,...,Green,SNP,PC,DUP,SF,SDLP,UUP,APNI,All other candidates,Of which other winner
0,W07000081,W92000004,Aberafan Maesteg,NaN,Wales,Wales,County,NaN,Stephen,Kinnock,...,1094,0,4719,0,0,0,0,0,801,0
1,S14000060,S92000003,Aberdeen North,NaN,Scotland,Scotland,Burgh,NaN,Kirsty,Blackman,...,1275,14533,0,0,0,0,0,0,1269,0
2,S14000061,S92000003,Aberdeen South,NaN,Scotland,Scotland,Burgh,NaN,Stephen,Flynn,...,1609,15213,0,0,0,0,0,0,648,0
3,S14000062,S92000003,Aberdeenshire North and Moray East,NaN,Scotland,Scotland,County,NaN,Seamus,Logan,...,0,13455,0,0,0,0,0,0,0,0
4,S14000063,S92000003,Airdrie and Shotts,NaN,Scotland,Scotland,County,NaN,Kenneth,Stevenson,...,0,11324,0,0,0,0,0,0,1079,0


In [11]:
data['Result'].value_counts()

Result
Lab hold               193
Lab gain from Con      182
Con hold               120
LD gain from Con        60
Lab gain from SNP       36
SNP hold                 8
LD hold                  8
SF hold                  7
RUK gain from Con        5
DUP hold                 5
Ind gain from Lab        5
LD gain from SNP         4
SDLP hold                2
PC gain from Con         2
PC hold                  2
Green gain from Con      2
SNP gain from Con        1
Green hold               1
Green gain from Lab      1
Spk hold                 1
APNI gain from DUP       1
Con gain from Lab        1
TUV gain from DUP        1
Ind gain from APNI       1
UUP gain from DUP        1
Name: count, dtype: int64

In [12]:
gains = data[data['Result'].str.contains('gain', case=False, na=False)].copy()


In [13]:
gains[['Gaining party', 'Losing party']] = gains['Result'].str.extract(r"^(\S+) gain from (\S+)$")
gains[['Gaining party', 'Losing party']].head()

,Gaining party,Losing party
3,SNP,Con
4,Lab,SNP
5,Lab,Con
7,Lab,SNP
8,Lab,Con


# Alluvial chart

In [14]:
# Conventional UK party colours (nodes are labelled, so colour is not the only cue)
party_colors = {
    "Con": "#0087DC", "Lab": "#E4003B", "LD": "#FAA61A", "SNP": "#E6C200",
    "RUK": "#12B6CF", "Green": "#02A95B", "PC": "#005B54", "Ind": "#8C8C8C",
    "DUP": "#D46A4C", "APNI": "#C9A227", "TUV": "#0C3A6A", "UUP": "#48A5EE",
}
REGION_COLOR = "#B0B0B0"

def rgba(hex_color, alpha=0.45):
    h = hex_color.lstrip("#")
    r, g, b = (int(h[i:i + 2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{alpha})"

def build_sankey(data):
    """Three-stage alluvial (lost by -> gained by -> region) for any subset of `gains`."""
    # --- Nodes: three columns, each ordered by size ---
    losers  = data["Losing party"].value_counts()   # left
    gainers = data["Gaining party"].value_counts()  # middle
    regions = data["Region name"].value_counts()    # right

    # labels = (
    #     [f"{p} lost {n}" for p, n in losers.items()]
    #     + [f"{p} gained {n}" for p, n in gainers.items()]
    #     + [f"{r} ({n})" for r, n in regions.items()]
    # )
    labels = (
        list(losers.index)
        + list(gainers.index)
        + list(regions.index)
    )
    node_colors = (
        [party_colors.get(p, "#8C8C8C") for p in losers.index]
        + [party_colors.get(p, "#8C8C8C") for p in gainers.index]
        + [REGION_COLOR] * len(regions)
    )
    left_idx   = {p: i for i, p in enumerate(losers.index)}
    middle_idx = {p: i + len(losers) for i, p in enumerate(gainers.index)}
    right_idx  = {r: i + len(losers) + len(gainers) for i, r in enumerate(regions.index)}

    # --- Links ---
    lm = data.groupby(["Losing party", "Gaining party"]).size().reset_index(name="Seats")
    mr = data.groupby(["Gaining party", "Region name"]).size().reset_index(name="Seats")

    source = [left_idx[p] for p in lm["Losing party"]] + [middle_idx[p] for p in mr["Gaining party"]]
    target = [middle_idx[p] for p in lm["Gaining party"]] + [right_idx[r] for r in mr["Region name"]]
    value  = lm["Seats"].tolist() + mr["Seats"].tolist()
    # Colour every band by the gaining party, so you can follow it across both stages
    link_colors = [rgba(party_colors.get(p, "#8C8C8C")) for p in lm["Gaining party"]] + \
                  [rgba(party_colors.get(p, "#8C8C8C")) for p in mr["Gaining party"]]
    hover = (
        (lm["Losing party"] + " → " + lm["Gaining party"]).tolist()
        + (mr["Gaining party"] + " gains in " + mr["Region name"]).tolist()
    )

    fig = go.Figure(go.Sankey(
        arrangement="snap",
        node=dict(
            label=labels, color=node_colors, pad=12, thickness=14,
            line=dict(color="white", width=1),
            hovertemplate="%{label}: %{value} seats<extra></extra>",
        ),
        link=dict(
            source=source, target=target, value=value, color=link_colors,
            customdata=hover, hovertemplate="%{customdata}: %{value} seats<extra></extra>",
        ),
    ))
    for x, text, anchor in [(0.0, "<b>Lost by</b>", "left"),
                            (0.5, "<b>Gained by</b>", "center"),
                            (1.0, "<b>Region</b>", "right")]:
        fig.add_annotation(x=x, y=1.06, xref="paper", yref="paper", text=text,
                           showarrow=False, xanchor=anchor)
    fig.update_layout(
        title=dict(text=f"General Election 2024: How and Where {len(data)} Seats Changed Hands", font=dict(size=15)),
        font=dict(size=12), height=700,
        margin=dict(l=20, r=20, t=90, b=20), paper_bgcolor="white",
    )
    return fig

In [15]:
ALL = "All"

def choices(col):
    return [ALL] + gains[col].value_counts().index.tolist()   # biggest first

@interact(
    lost_by=Dropdown(options=choices("Losing party"), description="Lost by"),
    gained_by=Dropdown(options=choices("Gaining party"), description="Gained by"),
    region=Dropdown(options=choices("Region name"), description="Region"),
)
def show_alluvial(lost_by=ALL, gained_by=ALL, region=ALL):
    data = gains
    if lost_by != ALL:
        data = data[data["Losing party"] == lost_by]
    if gained_by != ALL:
        data = data[data["Gaining party"] == gained_by]
    if region != ALL:
        data = data[data["Region name"] == region]

    if data.empty:
        print("No seats changed hands for this combination of filters.")
        return
    build_sankey(data).show()


interactive(children=(Dropdown(description='Lost by', options=('All', 'Con', 'SNP', 'Lab', 'DUP', 'APNI'), val…

# Bar chart

In [16]:
def build_transfer_bars(lost_by=ALL, gained_by=ALL):
    """Horizontal bar chart of seats transferred, for one 'from' and/or 'to' party."""
    data = gains
    if lost_by != ALL:
        data = data[data["Losing party"] == lost_by]
    if gained_by != ALL:
        data = data[data["Gaining party"] == gained_by]
    if data.empty:
        return None

    # The bars show whichever side is still open:
    #   From X, to All  -> where X's seats went (one bar per gaining party)
    #   From All, to Y  -> where Y's seats came from (one bar per losing party)
    #   Both All        -> seats gained per party, split by who lost them
    #   Both chosen     -> that transfer broken down by region
    if lost_by != ALL and gained_by != ALL:
        bar_col, stack_col = "Region name", None
        title = f"{lost_by} → {gained_by}: {len(data)} seats, by region"
    elif lost_by != ALL:
        bar_col, stack_col = "Gaining party", None
        title = f"Where {lost_by}'s {len(data)} lost seats went"
    elif gained_by != ALL:
        bar_col, stack_col = "Losing party", None
        title = f"Where {gained_by}'s {len(data)} gained seats came from"
    else:
        bar_col, stack_col = "Gaining party", "Losing party"
        title = f"All {len(data)} seats that changed hands, by gaining party"

    order = data[bar_col].value_counts().index[::-1].tolist()   # biggest at the top
    fig = go.Figure()

    if stack_col is None:
        counts = data[bar_col].value_counts().reindex(order)
        colors = ([party_colors.get(p, "#8C8C8C") for p in order]
                  if bar_col != "Region name" else [party_colors.get(gained_by, REGION_COLOR)] * len(order))
        fig.add_bar(
            y=order, x=counts.values, orientation="h",
            marker=dict(color=colors, line=dict(color="white", width=2)),
            text=counts.values, textposition="outside", textfont=dict(color="#333"),
            hovertemplate="%{y}: %{x} seats<extra></extra>", showlegend=False,
        )
    else:
        # One stacked segment per losing party, coloured by that party
        totals = data[bar_col].value_counts().reindex(order)
        for loser in data[stack_col].value_counts().index:
            sub = data[data[stack_col] == loser][bar_col].value_counts().reindex(order, fill_value=0)
            fig.add_bar(
                y=order, x=sub.values, orientation="h", name=f"from {loser}",
                marker=dict(color=party_colors.get(loser, "#8C8C8C"), line=dict(color="white", width=2)),
                hovertemplate=f"%{{y}} gained %{{x}} from {loser}<extra></extra>",
            )
        # Total at the end of each bar
        fig.add_scatter(y=order, x=totals.values, mode="text", text=totals.values,
                        textposition="middle right", textfont=dict(color="#333"),
                        showlegend=False, hoverinfo="skip")
        fig.update_layout(barmode="stack", legend=dict(title="Lost by", traceorder="normal"))

    fig.update_layout(
        title=dict(text=title, font=dict(size=15)),
        xaxis=dict(title="Seats", gridcolor="#E5E5E5", zeroline=False),
        yaxis=dict(title=None),
        height=max(320, 40 * len(order) + 140),
        margin=dict(l=20, r=40, t=70, b=40),
        plot_bgcolor="white", paper_bgcolor="white", font=dict(size=12), bargap=0.25,
    )
    return fig


@interact(
    lost_by=Dropdown(options=choices("Losing party"), description="Lost by"),
    gained_by=Dropdown(options=choices("Gaining party"), description="Gained by"),
)
def show_transfers(lost_by=ALL, gained_by=ALL):
    fig = build_transfer_bars(lost_by, gained_by)
    if fig is None:
        print(f"No seats went from {lost_by} to {gained_by}.")
        return
    fig.show()


interactive(children=(Dropdown(description='Lost by', options=('All', 'Con', 'SNP', 'Lab', 'DUP', 'APNI'), val…